In [2]:
text = """
Data science is an exciting field of technology.
It combines mathematics statistics and programming.
Data science helps us understand large amounts of data.
Data scientists collect data from different sources.
Data cleaning is an important step in data science.
Exploratory data analysis helps us understand patterns.
Data visualization makes complex information easier to understand.
Python is one of the most popular programming languages in data science.
Libraries like Pandas and NumPy help in data analysis.
Machine learning allows computers to learn from data.
Supervised learning uses labeled training data.
Unsupervised learning discovers hidden patterns in data.
Deep learning uses neural networks to solve complex problems.
Artificial intelligence is transforming many industries.
Data science helps businesses make better decisions.
"""

In [3]:
text = text.lower()

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
# tokenization
words = text.split()

In [6]:
vocab = sorted(set(words))

In [7]:
vocab_size = len(vocab)
print(vocab_size)

80


In [8]:
word_to_id = {
    word : i
    for i, word in enumerate(vocab)
}

In [9]:
print(word_to_id)

{'allows': 0, 'amounts': 1, 'an': 2, 'analysis': 3, 'analysis.': 4, 'and': 5, 'artificial': 6, 'better': 7, 'businesses': 8, 'cleaning': 9, 'collect': 10, 'combines': 11, 'complex': 12, 'computers': 13, 'data': 14, 'data.': 15, 'decisions.': 16, 'deep': 17, 'different': 18, 'discovers': 19, 'easier': 20, 'exciting': 21, 'exploratory': 22, 'field': 23, 'from': 24, 'help': 25, 'helps': 26, 'hidden': 27, 'important': 28, 'in': 29, 'industries.': 30, 'information': 31, 'intelligence': 32, 'is': 33, 'it': 34, 'labeled': 35, 'languages': 36, 'large': 37, 'learn': 38, 'learning': 39, 'libraries': 40, 'like': 41, 'machine': 42, 'make': 43, 'makes': 44, 'many': 45, 'mathematics': 46, 'most': 47, 'networks': 48, 'neural': 49, 'numpy': 50, 'of': 51, 'one': 52, 'pandas': 53, 'patterns': 54, 'patterns.': 55, 'popular': 56, 'problems.': 57, 'programming': 58, 'programming.': 59, 'python': 60, 'science': 61, 'science.': 62, 'scientists': 63, 'solve': 64, 'sources.': 65, 'statistics': 66, 'step': 67, 

In [10]:
id_to_word = {
    i : word
    for word, i in word_to_id.items()
}

In [11]:
print(id_to_word)

{0: 'allows', 1: 'amounts', 2: 'an', 3: 'analysis', 4: 'analysis.', 5: 'and', 6: 'artificial', 7: 'better', 8: 'businesses', 9: 'cleaning', 10: 'collect', 11: 'combines', 12: 'complex', 13: 'computers', 14: 'data', 15: 'data.', 16: 'decisions.', 17: 'deep', 18: 'different', 19: 'discovers', 20: 'easier', 21: 'exciting', 22: 'exploratory', 23: 'field', 24: 'from', 25: 'help', 26: 'helps', 27: 'hidden', 28: 'important', 29: 'in', 30: 'industries.', 31: 'information', 32: 'intelligence', 33: 'is', 34: 'it', 35: 'labeled', 36: 'languages', 37: 'large', 38: 'learn', 39: 'learning', 40: 'libraries', 41: 'like', 42: 'machine', 43: 'make', 44: 'makes', 45: 'many', 46: 'mathematics', 47: 'most', 48: 'networks', 49: 'neural', 50: 'numpy', 51: 'of', 52: 'one', 53: 'pandas', 54: 'patterns', 55: 'patterns.', 56: 'popular', 57: 'problems.', 58: 'programming', 59: 'programming.', 60: 'python', 61: 'science', 62: 'science.', 63: 'scientists', 64: 'solve', 65: 'sources.', 66: 'statistics', 67: 'step', 

In [12]:
data = torch.tensor([word_to_id[word] for word in words], dtype=torch.long)
print(type(data))

<class 'torch.Tensor'>


In [13]:
# parameters
batch_size = 4
embedding_dim = 32

In [14]:
# input and target
def get_batch():
  start = torch.randint(
      len(data) - batch_size,(4,)
  )

  x = torch.stack([data[i: i+batch_size] for i in start])
  y = torch.stack([data[i+1: i+1+batch_size] for i in start])

  return x, y

In [15]:
class MiniModel(nn.Module):
  def __init__(self):
    super().__init__()

    # token embedding
    self.embedding = nn.Embedding(vocab_size, embedding_dim)

    # position embedding
    self.position_embedding = nn.Embedding(batch_size, embedding_dim)

    # transformer layer
    self.transformer = nn.TransformerEncoderLayer(
        nhead=4,
        d_model=embedding_dim,
        batch_first=True)

    # output layer
    self.linear = nn.Linear(embedding_dim, vocab_size)

  def forward(self,x):
    B , T = x.shape

    tokens = self.embedding(x)

    position = self.position_embedding(torch.arange(T))

    x = tokens + position

    # casual mask
    mask = torch.triu(
        torch.ones(T,T, device=x.device), diagonal=1
    )

    x = self.transformer(x, src_mask=mask)

    output = self.linear(x)

    return output

In [16]:
model = MiniModel()

In [17]:
# optimize
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [18]:
# training
for epoches in range(1000):
  x,y = get_batch()

  prediction = model(x)

  loss = F.cross_entropy(prediction.view(-1, vocab_size), y.view(-1))

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoches % 100 == 0:
    print(f"loss : {loss.item()}")


loss : 4.627645969390869
loss : 2.596106767654419
loss : 1.2341972589492798
loss : 1.0030019283294678
loss : 0.46050673723220825
loss : 0.2774992883205414
loss : 0.23604239523410797
loss : 0.09033825993537903
loss : 0.08504988253116608
loss : 0.07400882244110107


In [19]:
model.eval()

MiniModel(
  (embedding): Embedding(80, 32)
  (position_embedding): Embedding(4, 32)
  (transformer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (linear1): Linear(in_features=32, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=32, bias=True)
    (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (linear): Linear(in_features=32, out_features=80, bias=True)
)

In [20]:
user_input = input("Enter a word : ")

context = torch.tensor([[word_to_id[user_input]]])

with torch.no_grad():

    for i in range(10):

        prediction = model(context[:, -batch_size:])

        prediction = prediction[:, -1, :]

        probabilities = F.softmax(prediction, dim=-1)

        next_word = torch.multinomial(
            probabilities, num_samples=1
        )

        context = torch.cat([context, next_word], dim=1)


generated_text = " ".join(
    id_to_word[i] for i in context[0].tolist()
)

print(generated_text)

Enter a word : data
data science. step in data science. libraries like pandas and numpy
